# BAB IV — Pemodelan Random Forest
**Judul Penelitian:** Penerapan Algoritma Random Forest untuk Prediksi Keberlanjutan Studi Mahasiswa pada Program Studi Sistem Informasi UKRIDA

---
**Alur BAB IV:**
1. Import Library
2. Load Dataset Hasil Preprocessing
3. Pembangunan Model Random Forest
4. Pelatihan Model
5. Prediksi pada Data Testing
6. Evaluasi Model
7. Confusion Matrix
8. Feature Importance
9. Simpan Model

## 1. Import Library

In [ ]:
# ============================================================
# STEP 1 — Import Library
# Tujuan: Memuat seluruh library yang dibutuhkan untuk
#         pemodelan, evaluasi, visualisasi, dan penyimpanan model
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import pickle
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score,
    roc_curve
)

print('Library berhasil diimport.')

## 2. Load Dataset Hasil Preprocessing
> **Catatan:** Upload file `dataset_train.csv` dan `dataset_test.csv` hasil preprocessing BAB III ke Google Colab terlebih dahulu.
> Atau gunakan `dataset_preprocessed.csv` dan lakukan split ulang.

In [ ]:
# ============================================================
# STEP 2A — Upload file ke Google Colab (jalankan jika perlu)
# ============================================================

# from google.colab import files
# uploaded = files.upload()   # upload dataset_train.csv & dataset_test.csv

# Jika sudah ada di direktori, langsung load:
df_train = pd.read_csv('dataset_train.csv')
df_test  = pd.read_csv('dataset_test.csv')

print('Dataset training dan testing berhasil dimuat.')
print(f'Ukuran data training : {df_train.shape[0]} baris x {df_train.shape[1]} kolom')
print(f'Ukuran data testing  : {df_test.shape[0]} baris x {df_test.shape[1]} kolom')

In [ ]:
# ============================================================
# STEP 2B —     
# ============================================================

KOLOM_TARGET = 'Target_Keberlanjutan_Studi'
FITUR_X = [
    'Semester_Saat_Ini',
    'IPK_Saat_Ini',
    'IPS_Semester_Berjalan',
    'Total_SKS_Lulus',
    'Status_Keuangan',
    'Total_Poin_Soft_Skill',
    'Bimbingan_Pending',
    'Jumlah_Bimbingan_Semester_Berjalan'
]

X_train = df_train[FITUR_X]
y_train = df_train[KOLOM_TARGET]
X_test  = df_test[FITUR_X]
y_test  = df_test[KOLOM_TARGET]

print('Fitur (X) dan Target (y) berhasil dipisahkan.')
print(f'\nX_train : {X_train.shape[0]} baris x {X_train.shape[1]} kolom')
print(f'X_test  : {X_test.shape[0]} baris x {X_test.shape[1]} kolom')
print(f'y_train : {y_train.shape[0]} nilai')
print(f'y_test  : {y_test.shape[0]} nilai')

In [ ]:
# ============================================================
# STEP 2C — Tampilkan Data Training (5 baris pertama)
# [SCREENSHOT BAB IV — Konfirmasi data siap dimodelkan]
# ============================================================

print('5 Baris Pertama Data Training (X_train):')
print('-' * 50)
print(X_train.head().to_string())

print('\nDistribusi kelas pada data training (y_train):')
vc_train = y_train.value_counts()
print(f'  Kelas 0 (Melanjutkan Studi)    : {vc_train[0]} mahasiswa ({vc_train[0]/len(y_train)*100:.1f}%)')
print(f'  Kelas 1 (Tidak Melanjutkan)    : {vc_train[1]} mahasiswa ({vc_train[1]/len(y_train)*100:.1f}%)')

print('\nDistribusi kelas pada data testing (y_test):')
vc_test = y_test.value_counts()
print(f'  Kelas 0 (Melanjutkan Studi)    : {vc_test[0]} mahasiswa ({vc_test[0]/len(y_test)*100:.1f}%)')
print(f'  Kelas 1 (Tidak Melanjutkan)    : {vc_test[1]} mahasiswa ({vc_test[1]/len(y_test)*100:.1f}%)')

## 3. Pembangunan Model Random Forest

### Rekomendasi Parameter untuk Dataset ~400 Data

| Parameter | Nilai | Alasan |
|-----------|-------|--------|
| `n_estimators` | `100` | 100 pohon sudah stabil untuk dataset kecil; lebih dari 200 tidak signifikan |
| `max_depth` | `None` | Pohon tumbuh bebas; Random Forest sudah mencegah overfitting secara internal |
| `min_samples_split` | `5` | Minimal 5 sampel untuk split; mencegah overfitting di dataset kecil |
| `min_samples_leaf` | `2` | Setiap daun minimal 2 sampel; menjaga generalisasi |
| `max_features` | `'sqrt'` | Default terbaik untuk klasifikasi: akar dari jumlah fitur |
| `class_weight` | `'balanced'` | Menyeimbangkan bobot kelas 72:28 secara otomatis |
| `random_state` | `42` | Reprodusibilitas hasil penelitian |

In [ ]:
# ============================================================
# STEP 3 — Inisialisasi Model Random Forest
# Tujuan: Mendefinisikan arsitektur dan parameter model
#         yang akan digunakan untuk klasifikasi
# [SCREENSHOT BAB IV — Parameter model]
# ============================================================

rf_model = RandomForestClassifier(
    n_estimators      = 100,      # jumlah pohon keputusan
    max_depth         = None,     # kedalaman pohon tidak dibatasi
    min_samples_split = 5,        # min sampel untuk melakukan split
    min_samples_leaf  = 2,        # min sampel pada setiap daun pohon
    max_features      = 'sqrt',   # jumlah fitur saat split: sqrt(n_features)
    class_weight      = 'balanced', # penanganan ketidakseimbangan kelas
    random_state      = 42        # seed untuk reprodusibilitas
)

print('Model Random Forest berhasil diinisialisasi.')
print('\nParameter Model:')
params = rf_model.get_params()
param_penting = ['n_estimators','max_depth','min_samples_split',
                 'min_samples_leaf','max_features','class_weight','random_state']
for p in param_penting:
    print(f'  {p:<22}: {params[p]}')

## 4. Pelatihan Model

In [ ]:
# ============================================================
# STEP 4 — Melatih Model Menggunakan Data Training
# Tujuan: Model mempelajari pola dari X_train dan y_train
#         sehingga dapat membuat prediksi pada data baru
# ============================================================

rf_model.fit(X_train, y_train)

print('Model Random Forest berhasil dilatih.')
print(f'Jumlah pohon yang dibangun : {rf_model.n_estimators}')
print(f'Jumlah fitur yang digunakan: {rf_model.n_features_in_}')
print(f'Kelas yang dipelajari      : {rf_model.classes_.tolist()}')

In [ ]:
# ============================================================
# STEP 4B — Akurasi pada Data Training
# Tujuan: Melihat seberapa baik model belajar dari data latih
#         (sebagai referensi untuk mendeteksi overfitting)
# ============================================================

y_pred_train = rf_model.predict(X_train)
acc_train    = accuracy_score(y_train, y_pred_train)

print(f'Akurasi pada Data Training : {acc_train*100:.2f}%')
print('(Nilai ini wajar lebih tinggi dari data testing)')

## 5. Prediksi pada Data Testing

In [ ]:
# ============================================================
# STEP 5 — Prediksi pada Data Testing
# Tujuan: Menggunakan model yang sudah dilatih untuk
#         memprediksi kelas pada data yang belum pernah dilihat
# ============================================================

y_pred      = rf_model.predict(X_test)
y_pred_prob = rf_model.predict_proba(X_test)[:, 1]  # probabilitas kelas 1

print('Prediksi berhasil dilakukan pada data testing.')
print(f'\nJumlah data testing   : {len(y_test)}')
print(f'Prediksi Kelas 0      : {(y_pred == 0).sum()}')
print(f'Prediksi Kelas 1      : {(y_pred == 1).sum()}')
print(f'\n5 Contoh Hasil Prediksi (Aktual vs Prediksi):')
hasil = pd.DataFrame({'Aktual': y_test.values[:5], 'Prediksi': y_pred[:5]})
hasil['Keterangan'] = hasil.apply(
    lambda r: 'Benar' if r['Aktual'] == r['Prediksi'] else 'Salah', axis=1)
print(hasil.to_string(index=False))

## 6. Evaluasi Model

Evaluasi menggunakan 4 metrik utama:
- **Accuracy** — proporsi prediksi benar dari seluruh prediksi
- **Precision** — proporsi prediksi positif yang benar-benar positif
- **Recall** — proporsi kelas positif yang berhasil teridentifikasi
- **F1-Score** — rata-rata harmonis Precision dan Recall

In [ ]:
# ============================================================
# STEP 6A — Menghitung Metrik Evaluasi
# [SCREENSHOT BAB IV — Tabel metrik evaluasi]
# ============================================================

acc       = accuracy_score(y_test, y_pred)
prec_0    = precision_score(y_test, y_pred, pos_label=0)
prec_1    = precision_score(y_test, y_pred, pos_label=1)
rec_0     = recall_score(y_test, y_pred, pos_label=0)
rec_1     = recall_score(y_test, y_pred, pos_label=1)
f1_0      = f1_score(y_test, y_pred, pos_label=0)
f1_1      = f1_score(y_test, y_pred, pos_label=1)
f1_macro  = f1_score(y_test, y_pred, average='macro')
f1_weight = f1_score(y_test, y_pred, average='weighted')
auc       = roc_auc_score(y_test, y_pred_prob)

print('=' * 55)
print('  HASIL EVALUASI MODEL RANDOM FOREST')
print('=' * 55)
print(f'  Accuracy               : {acc*100:.2f}%')
print(f'  AUC-ROC                : {auc:.4f}')
print()
print(f'  {'Metrik':<24}  {'Kelas 0':>10}  {'Kelas 1':>10}')
print(f'  {'-'*46}')
print(f'  {'Precision':<24}  {prec_0*100:>9.2f}%  {prec_1*100:>9.2f}%')
print(f'  {'Recall':<24}  {rec_0*100:>9.2f}%  {rec_1*100:>9.2f}%')
print(f'  {'F1-Score':<24}  {f1_0*100:>9.2f}%  {f1_1*100:>9.2f}%')
print(f'  {'-'*46}')
print(f'  {'F1-Score (Macro)':<24}  {f1_macro*100:>9.2f}%')
print(f'  {'F1-Score (Weighted)':<24}  {f1_weight*100:>9.2f}%')
print('=' * 55)

In [ ]:
# ============================================================
# STEP 6B — Classification Report Lengkap
# [SCREENSHOT BAB IV — Classification report]
# ============================================================

print('Classification Report:')
print('-' * 55)
print(classification_report(
    y_test, y_pred,
    target_names=['Melanjutkan Studi (0)', 'Tidak Melanjutkan (1)']
))

## 7. Confusion Matrix

In [ ]:
# ============================================================
# STEP 7 — Confusion Matrix
# Tujuan: Menampilkan distribusi prediksi benar dan salah
#         untuk setiap kelas secara visual
# [SCREENSHOT BAB IV — Confusion Matrix]
# ============================================================

cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

print('Confusion Matrix:')
print(f'  True Negative  (TN) : {tn}  — Prediksi Melanjutkan, Aktual Melanjutkan')
print(f'  False Positive (FP) : {fp}  — Prediksi Tidak Melanjutkan, Aktual Melanjutkan')
print(f'  False Negative (FN) : {fn}  — Prediksi Melanjutkan, Aktual Tidak Melanjutkan')
print(f'  True Positive  (TP) : {tp}  — Prediksi Tidak Melanjutkan, Aktual Tidak Melanjutkan')

fig, ax = plt.subplots(figsize=(7, 5))
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=['Melanjutkan\nStudi (0)', 'Tidak\nMelanjutkan (1)']
)
disp.plot(
    ax=ax, cmap='Blues', colorbar=False,
    values_format='d'
)
ax.set_title(
    'Confusion Matrix — Random Forest\n'
    'Prediksi Keberlanjutan Studi Mahasiswa UKRIDA',
    fontsize=12, fontweight='bold', pad=15
)
ax.set_xlabel('Prediksi', fontweight='bold', fontsize=11)
ax.set_ylabel('Aktual', fontweight='bold', fontsize=11)

# Tambahkan label TN/FP/FN/TP
labels = [['TN', 'FP'], ['FN', 'TP']]
for i in range(2):
    for j in range(2):
        ax.text(j, i - 0.25, labels[i][j],
                ha='center', va='center',
                fontsize=9, color='gray', style='italic')

plt.tight_layout()
plt.savefig('bab4_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Confusion matrix disimpan: bab4_confusion_matrix.png')

## 8. Feature Importance

In [ ]:
# ============================================================
# STEP 8A — Hitung Feature Importance
# Tujuan: Mengetahui kontribusi setiap fitur dalam
#         proses pengambilan keputusan model
# [SCREENSHOT BAB IV — Tabel feature importance]
# ============================================================

importances = rf_model.feature_importances_
fi_df = pd.DataFrame({
    'Fitur'       : FITUR_X,
    'Importance'  : importances,
    'Persentase'  : importances * 100
}).sort_values('Importance', ascending=False).reset_index(drop=True)
fi_df.index = fi_df.index + 1  # mulai dari 1
fi_df['Persentase'] = fi_df['Persentase'].round(2)
fi_df['Importance'] = fi_df['Importance'].round(4)

print('Feature Importance — Random Forest:')
print('-' * 55)
print(f'  {'Rank':<5} {'Fitur':<42} {'Importance':>10}  {'%':>7}')
print(f'  {'-'*52}')
for rank, row in fi_df.iterrows():
    bar = '█' * int(row['Importance'] * 50)
    print(f'  {rank:<5} {row["Fitur"]:<42} {row["Importance"]:>10.4f}  {row["Persentase"]:>6.2f}%')

In [ ]:
# ============================================================
# STEP 8B — Visualisasi Feature Importance (Bar Chart)
# [SCREENSHOT BAB IV — Grafik feature importance]
# ============================================================

# Warna berdasarkan peringkat (gradasi biru)
n = len(fi_df)
colors = plt.cm.Blues(np.linspace(0.35, 0.90, n))[::-1]

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(
    fi_df['Fitur'][::-1],
    fi_df['Importance'][::-1],
    color=colors, edgecolor='white', height=0.65
)

# Label nilai di ujung bar
for bar, imp, pct in zip(bars,
                          fi_df['Importance'][::-1],
                          fi_df['Persentase'][::-1]):
    ax.text(
        bar.get_width() + 0.002,
        bar.get_y() + bar.get_height() / 2,
        f'{imp:.4f}  ({pct:.1f}%)',
        va='center', fontsize=9
    )

ax.set_xlabel('Feature Importance Score', fontweight='bold', fontsize=11)
ax.set_title(
    'Feature Importance — Random Forest\n'
    'Prediksi Keberlanjutan Studi Mahasiswa UKRIDA',
    fontsize=12, fontweight='bold', pad=12
)
ax.set_xlim(0, fi_df['Importance'].max() * 1.30)
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='x', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.savefig('bab4_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Feature importance chart disimpan: bab4_feature_importance.png')

## 8C. Kurva ROC-AUC *(opsional untuk BAB IV)*

In [ ]:
# ============================================================
# STEP 8C — Kurva ROC-AUC (opsional)
# Tujuan: Menampilkan kemampuan diskriminasi model pada
#         berbagai threshold klasifikasi
# [SCREENSHOT BAB IV — Kurva ROC (opsional)]
# ============================================================

fpr, tpr, thresholds = roc_curve(y_test, y_pred_prob)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(fpr, tpr, color='#1565C0', lw=2,
        label=f'ROC Curve (AUC = {auc:.3f})')
ax.plot([0, 1], [0, 1], color='gray', linestyle='--', lw=1, label='Baseline')
ax.fill_between(fpr, tpr, alpha=0.08, color='#1565C0')
ax.set_xlabel('False Positive Rate', fontweight='bold')
ax.set_ylabel('True Positive Rate', fontweight='bold')
ax.set_title('Kurva ROC — Random Forest\nPrediksi Keberlanjutan Studi Mahasiswa UKRIDA',
             fontsize=12, fontweight='bold')
ax.legend(loc='lower right', fontsize=10)
ax.set_xlim([0, 1]); ax.set_ylim([0, 1.02])
ax.spines[['top', 'right']].set_visible(False)
ax.grid(linestyle='--', alpha=0.3)

plt.tight_layout()
plt.savefig('bab4_roc_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Kurva ROC disimpan: bab4_roc_curve.png')
print(f'AUC-ROC Score: {auc:.4f}')

## 9. Simpan Model

In [ ]:
# ============================================================
# STEP 9 — Simpan Model ke File .sav
# Tujuan: Menyimpan model yang sudah dilatih menggunakan
#         pickle agar dapat digunakan kembali tanpa melatih ulang
#         (untuk implementasi Streamlit di tahap berikutnya)
# [SCREENSHOT BAB IV — Konfirmasi penyimpanan model]
# ============================================================

NAMA_FILE_MODEL = 'random_forest_ukrida.sav'

with open(NAMA_FILE_MODEL, 'wb') as f:
    pickle.dump(rf_model, f)

print(f'Model berhasil disimpan sebagai: {NAMA_FILE_MODEL}')

In [ ]:
# ============================================================
# STEP 9B — Verifikasi Model yang Disimpan
# Tujuan: Memastikan model dapat dimuat kembali dan
#         menghasilkan prediksi yang identik
# ============================================================

# Load model dari file
with open(NAMA_FILE_MODEL, 'rb') as f:
    model_loaded = pickle.load(f)

# Uji prediksi dengan model yang sudah dimuat
y_pred_verify = model_loaded.predict(X_test)
acc_verify    = accuracy_score(y_test, y_pred_verify)

print('Verifikasi Model:')
print(f'  File model          : {NAMA_FILE_MODEL}')
print(f'  Status load         : Berhasil')
print(f'  Akurasi verifikasi  : {acc_verify*100:.2f}%')
print(f'  Konsistensi         : {"Identik" if acc_verify == acc else "Berbeda"}')
print(f'\nModel siap digunakan untuk implementasi Streamlit (BAB IV/V).')

## 10. Ringkasan Hasil Pemodelan

In [ ]:
# ============================================================
# STEP 10 — Ringkasan Hasil Pemodelan
# [SCREENSHOT BAB IV — Ringkasan hasil keseluruhan]
# ============================================================

print('=' * 58)
print('  RINGKASAN HASIL PEMODELAN RANDOM FOREST — UKRIDA')
print('=' * 58)
print(f'  Dataset          : Keberlanjutan Studi Mahasiswa SI UKRIDA')
print(f'  Total data       : {len(y_train)+len(y_test)} mahasiswa (angkatan 2013–2025)')
print(f'  Data training    : {len(y_train)} baris (80%)')
print(f'  Data testing     : {len(y_test)} baris (20%)')
print(f'  Jumlah fitur     : {X_train.shape[1]} fitur')
print()
print(f'  Parameter Utama:')
print(f'    n_estimators   : 100')
print(f'    max_features   : sqrt')
print(f'    class_weight   : balanced')
print(f'    random_state   : 42')
print()
print(f'  Hasil Evaluasi:')
print(f'    Accuracy       : {acc*100:.2f}%')
print(f'    Precision (0)  : {prec_0*100:.2f}%   Precision (1): {prec_1*100:.2f}%')
print(f'    Recall    (0)  : {rec_0*100:.2f}%   Recall    (1): {rec_1*100:.2f}%')
print(f'    F1-Score  (0)  : {f1_0*100:.2f}%   F1-Score  (1): {f1_1*100:.2f}%')
print(f'    F1 Weighted    : {f1_weight*100:.2f}%')
print(f'    AUC-ROC        : {auc:.4f}')
print()
print(f'  Fitur Terpenting   : {fi_df.iloc[0]["Fitur"]} ({fi_df.iloc[0]["Persentase"]:.2f}%)')
print(f'  Fitur Ke-2         : {fi_df.iloc[1]["Fitur"]} ({fi_df.iloc[1]["Persentase"]:.2f}%)')
print(f'  Fitur Ke-3         : {fi_df.iloc[2]["Fitur"]} ({fi_df.iloc[2]["Persentase"]:.2f}%)')
print()
print(f'  File model disimpan: random_forest_ukrida.sav')
print('=' * 58)